<a href="https://colab.research.google.com/github/kuds/rl-atari-breakout/blob/main/%5BAtari%20Breakout%5D%20Single-Agent%20Reinforcement%20Learning%20DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Atari Breakout] Single-Agent Reinforcement Learning DQN

In [ ]:
!pip install swig

In [ ]:
!pip install stable-baselines3

In [ ]:
!pip install gymnasium[atari] ale-py

In [ ]:
import os
import csv
import time
import json
import platform
import torch
import numpy
import ale_py
import shutil
import stable_baselines3
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import VecVideoRecorder
from stable_baselines3.common.vec_env import VecTransposeImage
import matplotlib.pyplot
import matplotlib
import gymnasium
from importlib.metadata import version
from datetime import datetime
import google.colab.drive

In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
if torch.cuda.is_available(): print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Scipy Version: {version('scipy')}")
print(f"Swig Version: {version('swig')}")
print(f"Stable Baselines3 Version: {version('stable_baselines3')}")
print(f"ALE Version: {version('ale_py')}")

In [ ]:
rl_type = "DQN"
env_str = "ALE/Breakout-v5"
# ALE env IDs contain a slash; sanitize for filesystem paths.
env_slug = env_str.replace("/", "-")
name_prefix = f"{env_slug}_{rl_type}".lower()
log_dir = ""
parent_path = ""
use_google_drive = True
if use_google_drive:
    parent_path = "/content/gdrive"
    google.colab.drive.mount(parent_path, force_remount=True)
    log_dir = "{}/MyDrive/Finding Theta/rl-atari-breakout/logs/{}/{}".format(parent_path,
                                                       env_slug,
                                                       rl_type)
else:
    log_dir = "/content/logs/{}/{}".format(env_slug, rl_type)

In [ ]:
# Wrapper / env kwargs for ALE/Breakout-v5.
#
# v5 ships with frameskip=4 and sticky actions (repeat_action_probability=0.25).
# We disable both at the env level and let stable-baselines3's AtariWrapper
# do frame skipping (frame_skip=4), so we don't double-frameskip.
#
# https://danieltakeshi.github.io/2016/11/25/frame-skipping-and-preprocessing-for-deep-q-networks-on-atari-2600-games/
# https://stable-baselines3.readthedocs.io/en/master/common/atari_wrappers.html

env_kwargs = {
    "render_mode": "rgb_array",
    "frameskip": 1,
    "repeat_action_probability": 0.0,
}
wrapper_kwargs = {"frame_skip": 4}
# Eval env adds noop_max for random no-op starts at episode beginnings.
eval_wrapper_kwargs = {**wrapper_kwargs, "noop_max": 30}

In [ ]:
training_data_path = os.path.join(log_dir, "training jobs")
time_folder = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
model_folder_path = os.path.join(log_dir, "training jobs", time_folder)

In [ ]:
#Create Folders
os.makedirs(log_dir, exist_ok=True)
os.makedirs(training_data_path, exist_ok=True)
os.makedirs(model_folder_path, exist_ok=True)

In [ ]:
env = make_atari_env(env_str,
                     n_envs=1,
                     seed=0,
                     env_kwargs=env_kwargs,
                     wrapper_kwargs=wrapper_kwargs)
print("Observation Space Size: ", env.observation_space.shape)
print('Actions Space: ', env.action_space)
env.close()

In [ ]:
env = gymnasium.make(env_str)
print("Observation Space Size: ", env.observation_space.shape)
print('Actions Space: ', env.action_space)
env.close()

In [ ]:
# Use Hyperparameters from RL Zoo
hyperparams = {
    "env_str": env_str,
    "rl_type": rl_type,
    "eval_freq": 250_000,
    "checkpoint_freq": 250_000,
    "video_freq": 250_000,
    "buffer_size": 100_000,
    "total_timesteps": 5_000_000,
    "train_freq": 4,
    "target_update_interval": 1_000,
    "gradient_steps": 1,
    "exploration_fraction": 0.1,
    "exploration_final_eps": 0.01,
    "n_envs": 1,
    "batch_size": 32,
    "learning_rate":0.0001,
    "learning_starts":100_000,
    "optimize_memory_usage": False,
}

# atari:
#   env_wrapper:
#     - stable_baselines3.common.atari_wrappers.AtariWrapper
#   frame_stack: 4
#   policy: 'CnnPolicy'
#   n_timesteps: !!float 1e7
#   buffer_size: 100000
#   learning_rate: !!float 1e-4
#   batch_size: 32
#   learning_starts: 100000
#   target_update_interval: 1000
#   train_freq: 4
#   gradient_steps: 1
#   exploration_fraction: 0.1
#   exploration_final_eps: 0.01
#   # If True, you need to deactivate handle_timeout_termination
#   # in the replay_buffer_kwargs
#   optimize_memory_usage: False

In [ ]:
with open(os.path.join(model_folder_path, 'parameters.json'), 'w') as fp:
    parameters = {"hyperparams": hyperparams,
                  "env_kwargs": env_kwargs,
                  "wrapper_kwargs": wrapper_kwargs,
                  "eval_wrapper_kwargs": eval_wrapper_kwargs}
    json.dump(parameters, fp, indent=4)

In [ ]:
class VideoRecordCallback(BaseCallback):
    def __init__(
        self,
        save_path: str,
        video_length: int,
        save_freq: int = 5_000,
        name_prefix: str ="rl_model",
        verbose: int = 0):

        super().__init__(verbose)
        self.save_freq = save_freq
        self.video_length = video_length
        self.save_path = save_path
        self.name_prefix = name_prefix
        # Those variables will be accessible in the callback
        # (they are defined in the base class)
        # The RL model
        # self.model = None  # type: BaseAlgorithm
        # An alias for self.model.get_env(), the environment used for training
        # self.training_env # type: VecEnv
        # Number of time the callback was called
        # self.n_calls = 0  # type: int
        # num_timesteps = n_envs * n times env.step() was called
        # self.num_timesteps = 0  # type: int
        # local and global variables
        # self.locals = {}  # type: Dict[str, Any]
        # self.globals = {}  # type: Dict[str, Any]
        # The logger object, used to report things in the terminal
        # self.logger # type: stable_baselines3.common.logger.Logger
        # Sometimes, for event callback, it is useful
        # to have access to the parent object
        # self.parent = None  # type: Optional[BaseCallback]

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:

          name_prefix = f"{self.name_prefix}_{self.num_timesteps}"

          # Record video of the best model playing Atari's Environment
          rec_val = make_atari_env(
              env_str,
              n_envs=1,
              seed=1,
              env_kwargs=env_kwargs,
              wrapper_kwargs=eval_wrapper_kwargs)
          rec_val = VecFrameStack(rec_val, n_stack=4)
          rec_val = VecTransposeImage(rec_val)
          rec_val = VecVideoRecorder(rec_val,
                                    self.save_path,
                                    video_length=self.video_length,
                                    record_video_trigger=lambda x: x == 0,
                                    name_prefix=name_prefix)

          obs = rec_val.reset()
          for _ in range(self.video_length):
              action, _states = self.model.predict(obs)
              obs, rewards, dones, info = rec_val.step(action)
              rec_val.render()
              if dones:
                break

          rec_val.close()
        return True

In [ ]:
# Create the Training Atari's Breakout environment with appropriate wrappers
env = make_atari_env(env_str,
                     n_envs=hyperparams["n_envs"],
                     seed=0,
                     monitor_dir=os.path.join(model_folder_path, "monitor"),
                     env_kwargs=env_kwargs,
                     wrapper_kwargs=wrapper_kwargs)
env = VecFrameStack(env, n_stack=4)
env = VecTransposeImage(env)

# Create the Evaluation Atari's Breakout environment with appropriate wrappers
env_val = make_atari_env(env_str,
                         n_envs=1,
                         seed=1,
                         env_kwargs=env_kwargs,
                         wrapper_kwargs=eval_wrapper_kwargs)
env_val = VecFrameStack(env_val, n_stack=4)
env_val = VecTransposeImage(env_val)

In [ ]:
# Create Callbacks
# Create Evaluation Callback
# eval_freq - can cause learning instability if set to low
eval_callback = EvalCallback(
    env_val,
    best_model_save_path=model_folder_path,
    log_path=model_folder_path,
    eval_freq=hyperparams["eval_freq"],
    render=False,
    deterministic=True,
    n_eval_episodes=5)

# Create Checkpoint Callback
checkpoint_callback = CheckpointCallback(
    save_freq=hyperparams["checkpoint_freq"],
    save_path=os.path.join(model_folder_path, "checkpoints"),
    name_prefix=name_prefix,
    save_replay_buffer=False,
    save_vecnormalize=False,
)

video_record_callback = VideoRecordCallback(
    save_path=os.path.join(model_folder_path, "videos"),
    video_length=2_500,
    save_freq=hyperparams["video_freq"],
    name_prefix=name_prefix)

# Create the callback list
callbackList = CallbackList([checkpoint_callback,
                             video_record_callback,
                             eval_callback])

In [ ]:
# Initialize DQN
model = DQN('CnnPolicy',
            env,
            verbose=0,
            buffer_size=hyperparams["buffer_size"],
            batch_size=hyperparams["batch_size"],
            learning_rate=hyperparams["learning_rate"],
            learning_starts=hyperparams["learning_starts"],
            train_freq=hyperparams["train_freq"],
            gradient_steps=hyperparams["gradient_steps"],
            target_update_interval=hyperparams["target_update_interval"],
            exploration_fraction=hyperparams["exploration_fraction"],
            exploration_final_eps=hyperparams["exploration_final_eps"],
            optimize_memory_usage=hyperparams["optimize_memory_usage"],
            tensorboard_log=os.path.join(log_dir, "tensorboard"))

# Train the model
training_start_time = time.time()
model.learn(total_timesteps=hyperparams["total_timesteps"],
            progress_bar=False,
            callback=callbackList)
training_end_time = time.time()
training_duration_seconds = training_end_time - training_start_time

# Save the trained model
model.save(os.path.join(model_folder_path, "final_model"))

env.close()
env_val.close()

In [ ]:
# # Get model path from last training job (uncomment if training job interrupted)
# # List all entries in the directory
# entries = os.listdir(training_data_path)

# # Filter out only directories
# folders = [entry for entry in entries if os.path.isdir(os.path.join(training_data_path, entry))]

# # Sort the folders alphabetically
# folders.sort()

# # Get the last folder
# model_folder_path = os.path.join(training_data_path, folders[-1])
# print(model_folder_path)

## Elevate Best Model and Record Playback

In [ ]:
# Create the Evaluation Atari's Breakout environment with appropriate wrappers
env_val = make_atari_env(env_str,
                         n_envs=1,
                         seed=1,
                         env_kwargs=env_kwargs,
                         wrapper_kwargs=eval_wrapper_kwargs)
env_val = VecFrameStack(env_val, n_stack=4)
env_val = VecTransposeImage(env_val)

# Load the best model
best_model_path = os.path.join(model_folder_path, "best_model")
best_model = DQN.load(best_model_path, env=env_val)

episode_rewards, episode_lengths = evaluate_policy(
    best_model,
    env_val,
    n_eval_episodes=5,
    return_episode_rewards=True)
mean_reward = float(numpy.mean(episode_rewards))
std_reward = float(numpy.std(episode_rewards))
mean_ep_length = float(numpy.mean(episode_lengths))
std_ep_length = float(numpy.std(episode_lengths))
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

best_metrics_path = os.path.join(log_dir, "best_model_metrics.csv")

# Create Best Model Metrics file if not there
if(not os.path.isfile(best_metrics_path)):
  with open(best_metrics_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["run_date",
                     "batch_size",
                     "learning_rate",
                     "num_timesteps",
                     "mean_reward",
                     "std_reward",
                     "n_envs",
                     "gamma",
                     "seed"])

new_data = [os.path.basename(os.path.normpath(model_folder_path)),
            best_model.batch_size,
            best_model.learning_rate,
            best_model.num_timesteps,
            mean_reward,
            std_reward,
            best_model.n_envs,
            best_model.gamma,
            best_model.seed]

with open(best_metrics_path, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(new_data)

# Record video of the best model playing Atari's Breakout
best_model_file_name = "best_model_{}".format(name_prefix)
rec_val = VecVideoRecorder(env_val,
                           os.path.join(model_folder_path, "videos"),
                           video_length=10_000,
                           record_video_trigger=lambda x: x == 0,
                           name_prefix=best_model_file_name)

obs = rec_val.reset()
for _ in range(10_000):
    action, _states = best_model.predict(obs)
    obs, rewards, dones, info = rec_val.step(action)
    rec_val.render()
    if dones:
      break

env_val.close()
rec_val.close()

In [ ]:
# Print Model
print(best_model.policy)

In [ ]:
# Load the evaluations.npz file
data = numpy.load(os.path.join(model_folder_path, "evaluations.npz"))

# Extract the relevant data
timesteps = data['timesteps']
results = data['results']

# Calculate the mean and standard deviation of the results
mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

# Plot the results
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"{rl_type} Performance on {env_str}")
matplotlib.pyplot.savefig(os.path.join(model_folder_path, f"{rl_type}_{env_slug}_performance.png"))
matplotlib.pyplot.show()

In [ ]:
# Stage Summary
def _format_duration(seconds):
    seconds = int(round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


_summary_title = f"Atari Breakout: {rl_type} Training Summary"
_separator = "=" * 50

print(_summary_title)
print(_separator)
print()
print(f"Environment:    {env_str}")
print(f"Algorithm:      {rl_type}")
print(f"Date:           {time_folder}")
print(f"Timesteps:      {hyperparams['total_timesteps']:,}")
try:
    _duration_str = _format_duration(training_duration_seconds)
except NameError:
    _duration_str = "N/A (training cell not run in this session)"
print(f"Duration:       {_duration_str}")
print(f"Eval reward:    {mean_reward:.2f} +/- {std_reward:.2f}")
try:
    print(f"Avg ep length:  {mean_ep_length:.1f} +/- {std_ep_length:.1f} steps")
except NameError:
    pass
print(f"Best model:     {best_model_path}.zip")


In [ ]:
# Create version execution
notebook_name = "notebook.ipynb"
%notebook -e $notebook_name

In [ ]:
source_file = os.path.join(notebook_name)
destination_file = os.path.join(model_folder_path, notebook_name)

try:
    shutil.copyfile(notebook_name, destination_file)
    print(f"File '{source_file}' copied to '{destination_file}' successfully.")
except FileNotFoundError:
    print(f"Error: Source file '{source_file}' not found.")
except shutil.SameFileError:
    print(f"Error: Source and destination are the same file.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()